# Lab 3B — From vibration signals to spectral features

**Audience:** beginner Strategic Engineering MSc students  
**Scope:** Lecture 3, slides 69–95  
**Estimated time:** 60–75 min

## Scenario
A rotating machine is monitored by a vibration sensor sampled at **100 Hz**. We will work with short synthetic vibration windows representing two operating conditions:

- **normal:** a dominant rotational component around 10 Hz with weak harmonics;
- **fault-like:** the same main rotation plus stronger harmonics and more high-frequency energy.

The goal is **not** to build a sophisticated classifier. The goal is to learn how a time signal becomes a frequency-domain representation and how to construct interpretable spectral features that could later be used by a conventional ML model.

## Learning objectives
By the end of the lab you should be able to:

1. relate sampling frequency, sampling interval and Nyquist frequency;
2. compute and interpret a one-sided FFT spectrum;
3. connect window duration to frequency resolution;
4. extract dominant-frequency and band-power features;
5. explain spectral leakage and the role of a Hann window;
6. combine time-domain and frequency-domain features into a feature table;
7. recognize what information manual features preserve and what they can discard.


## 0. Setup
Run the following cells. The dataset is synthetic so that the lab is fully self-contained.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

FS = 100.0               # Hz
DT = 1.0 / FS            # seconds
WINDOW_SECONDS = 4.0
N = int(FS * WINDOW_SECONDS)
t = np.arange(N) / FS

print(f"Sampling frequency: {FS:.0f} Hz")
print(f"Sampling interval: {DT:.3f} s")
print(f"Frames per window: {N}")
print(f"Window duration: {N/FS:.1f} s")


In [ ]:
def make_vibration_window(condition, rng, fs=FS, n=N):
    # Create one synthetic vibration window for teaching purposes.
    time = np.arange(n) / fs

    # Slight run-to-run variability
    f0 = rng.normal(10.0, 0.20)
    amp = rng.normal(1.0, 0.08)
    phase = rng.uniform(0, 2*np.pi)

    x = amp * np.sin(2*np.pi*f0*time + phase)

    if condition == "normal":
        x += 0.12 * np.sin(2*np.pi*(2*f0)*time + 0.3)
        x += 0.05 * np.sin(2*np.pi*35.0*time + 1.0)
        noise = 0.18
    elif condition == "fault-like":
        # Stronger harmonics and more high-frequency content.
        x += 0.38 * np.sin(2*np.pi*(2*f0)*time + 0.3)
        x += 0.25 * np.sin(2*np.pi*(3*f0)*time + 0.8)
        x += 0.20 * np.sin(2*np.pi*35.0*time + 1.0)
        noise = 0.22
    else:
        raise ValueError("condition must be 'normal' or 'fault-like'")

    # Add a small DC offset that varies by recording.
    x += rng.normal(0.15, 0.04)
    x += rng.normal(0.0, noise, size=n)
    return x

records = []
for condition in ["normal", "fault-like"]:
    for i in range(60):
        records.append({
            "condition": condition,
            "window_id": f"{condition}_{i:02d}",
            "signal": make_vibration_window(condition, rng),
        })

print(f"Generated {len(records)} windows.")


In [ ]:
normal = next(r["signal"] for r in records if r["condition"] == "normal")
fault = next(r["signal"] for r in records if r["condition"] == "fault-like")

plt.figure(figsize=(11, 4))
plt.plot(t, normal, label="normal", alpha=0.9)
plt.plot(t, fault, label="fault-like", alpha=0.75)
plt.xlabel("Time [s]")
plt.ylabel("Vibration amplitude [a.u.]")
plt.title("Two example vibration windows")
plt.legend()
plt.show()


## 1. Sampling frequency and Nyquist limit

For **Fs = 100 Hz**:

1. compute the sampling interval Δt;
2. compute the Nyquist frequency;
3. explain what happens to a physical frequency above the Nyquist limit after sampling.

Complete the code below.

In [ ]:
sampling_interval = ...
nyquist = ...

print("Sampling interval [s] =", sampling_interval)
print("Nyquist frequency [Hz] =", nyquist)


**Question:** Can a 65 Hz vibration component be uniquely represented when Fs = 100 Hz? Why?

## 2. From time domain to frequency domain

We first remove the mean to avoid a strong DC component at 0 Hz. Then we compute the one-sided FFT using `np.fft.rfft`.

Complete the helper function.

In [ ]:
def power_spectrum(x, fs=FS, apply_hann=False):
    x = np.asarray(x)
    x = x - np.mean(x)

    if apply_hann:
        x = x * np.hanning(len(x))

    # TODO: compute the one-sided FFT
    X = ...

    # A simple relative power representation is sufficient for this lab.
    power = np.abs(X) ** 2

    # TODO: construct the physical frequency axis in Hz
    freq = ...
    return freq, power

freq_n, power_n = power_spectrum(normal)
freq_f, power_f = power_spectrum(fault)


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(freq_n, power_n, label="normal")
plt.plot(freq_f, power_f, label="fault-like", alpha=0.8)
plt.xlim(0, FS/2)
plt.xlabel("Frequency [Hz]")
plt.ylabel("Power [a.u.]")
plt.title("One-sided power spectra")
plt.legend()
plt.show()


**Questions:** Which component is dominant? Where do you see harmonics? Which condition contains more energy at higher frequencies?

## 3. Dominant frequency

The dominant frequency is the frequency with maximum spectral power. For these data, ignore the 0 Hz bin.

Complete the function.

In [ ]:
def dominant_frequency(x, fs=FS):
    freq, power = power_spectrum(x, fs)
    # TODO: ignore the zero-frequency bin and find the maximum-power bin
    idx = ...
    return ...

print("Normal dominant frequency:", dominant_frequency(normal), "Hz")
print("Fault-like dominant frequency:", dominant_frequency(fault), "Hz")


## 4. Frequency resolution and window duration

The approximate frequency-bin spacing is:

**Δf = Fs / L**

where L is the number of frames in the analysed window.

Compute the resolution for 1 s and 4 s windows at 100 Hz.

In [ ]:
resolution_1s = ...
resolution_4s = ...

print("1 s window resolution:", resolution_1s, "Hz")
print("4 s window resolution:", resolution_4s, "Hz")


In [ ]:
# Two nearby sinusoidal components: 10.0 and 10.8 Hz

def two_tone_signal(duration_s, fs=FS):
    tt = np.arange(int(duration_s*fs)) / fs
    xx = np.sin(2*np.pi*10.0*tt) + 0.8*np.sin(2*np.pi*10.8*tt)
    return tt, xx

# TODO: compute and plot the spectra for 1 s and 4 s windows.
# Which duration separates the two frequencies more clearly?


## 5. Band power and relative band power

For this teaching example use these bands:

- low: 0–15 Hz
- mid: 15–30 Hz
- high: 30–50 Hz

Band power is the sum of spectral power in a frequency interval. Relative band power divides that value by total spectral power.

In [ ]:
BANDS = {
    "low": (0, 15),
    "mid": (15, 30),
    "high": (30, 50),
}

def band_power(x, f_low, f_high, fs=FS, relative=False):
    freq, power = power_spectrum(x, fs)
    mask = (freq >= f_low) & (freq < f_high)

    # TODO: sum power in the selected band
    bp = ...

    if relative:
        # TODO: divide by total power; add a tiny number for numerical safety
        bp = ...
    return bp

for name, (lo, hi) in BANDS.items():
    print(name, "normal =", band_power(normal, lo, hi, relative=True),
          "fault-like =", band_power(fault, lo, hi, relative=True))


**Question:** Which band appears most informative for distinguishing the two conditions?

## 6. Spectral leakage and Hann window

A finite window can cause energy from a sinusoid to spread across neighbouring frequency bins, especially when the window does not contain an integer number of cycles.

Use a **10.5 Hz** sinusoid in a 1 s window and compare the spectrum without and with a Hann window.

In [ ]:
tt = np.arange(int(FS)) / FS
leak_signal = np.sin(2*np.pi*10.5*tt)

# TODO: compute spectra with apply_hann=False and apply_hann=True
freq_raw, power_raw = ...
freq_hann, power_hann = ...

# Normalize only for visual comparison.
power_raw = power_raw / power_raw.max()
power_hann = power_hann / power_hann.max()

plt.figure(figsize=(10, 4))
plt.plot(freq_raw, power_raw, marker="o", label="no window")
plt.plot(freq_hann, power_hann, marker="o", label="Hann window")
plt.xlim(4, 18)
plt.xlabel("Frequency [Hz]")
plt.ylabel("Normalized power")
plt.title("Spectral leakage: finite window vs Hann window")
plt.legend()
plt.show()


**Question:** What improves with the Hann window? What is the trade-off around the main peak?

## 7. Build a combined feature table

Now transform each vibration window into a compact feature vector. Include both time-domain and frequency-domain information.

Suggested features:

- mean
- standard deviation
- RMS
- maximum absolute value
- dominant frequency
- relative low-, mid- and high-band power


In [ ]:
def rms(x):
    return np.sqrt(np.mean(np.asarray(x)**2))


def extract_features(x):
    # TODO: return a dictionary containing the requested features.
    return {
        "mean": ...,
        "std": ...,
        "rms": ...,
        "max_abs": ...,
        "dominant_freq": ...,
        "rel_low": ...,
        "rel_mid": ...,
        "rel_high": ...,
    }

rows = []
for r in records:
    row = extract_features(r["signal"])
    row["condition"] = r["condition"]
    row["window_id"] = r["window_id"]
    rows.append(row)

features = pd.DataFrame(rows)
features.head()


In [ ]:
# Compare average feature values by condition.
features.groupby("condition").mean(numeric_only=True).round(3)


In [ ]:
# Visualize two interpretable features.
for condition, group in features.groupby("condition"):
    plt.scatter(group["rms"], group["rel_high"], label=condition, alpha=0.7)

plt.xlabel("RMS vibration")
plt.ylabel("Relative high-band power")
plt.title("Feature space prepared for a later classifier")
plt.legend()
plt.show()


## 8. Engineering interpretation

Answer briefly:

1. Which feature(s) separate the two synthetic conditions most clearly?
2. Why might relative band power transfer better than absolute band power when overall vibration amplitude changes?
3. Why do we remove the mean before spectral analysis?
4. Why can a longer window improve frequency resolution but worsen temporal localization?
5. What temporal information is lost when a complete waveform is replaced by a small feature vector?


## Optional challenge — aliasing

Generate a 35 Hz sinusoid and sample it at **Fs = 60 Hz**. Since the Nyquist frequency is only 30 Hz, inspect the sampled spectrum and determine the lower frequency at which the component appears.

This is an example of **aliasing**.

In [ ]:
# Optional challenge
# TODO: create and inspect the aliased signal.


## Take-away

A frequency-domain pipeline makes explicit engineering assumptions:

**window → preprocessing → spectrum → selected spectral features**

This can be interpretable and effective, especially when the relevant physics is known. But the feature designer must decide which temporal scales, frequencies and summaries matter, and some sequence structure is inevitably discarded.
